# Train `ASCIIVAE` on Google Colab

End-to-end Colab notebook for training `world_model/model/net/ascii_vae.py::ASCIIVAE` using the trainer in `world_model/train_ascii_vae.py`.

## Runtime

1. `Runtime` → `Change runtime type` → **GPU** → pick **A100** or **H100** (requires Colab Pro / Pro+).
2. `Runtime` → `Change runtime type` → enable **High-RAM** if offered.

The ASCIIVAE is tiny (canvas 16×32, hidden 128), so any modern GPU works; H100 is overkill but fine.

## Data (one-time upload to Google Drive)

The repo's `.gitignore` excludes `data/transitions/`, so the training shards are **not** in the repo. Upload them to Drive **before** running this notebook.

**Fastest path for 1–10 GB (lots of small files):**

On your laptop, tar the transitions folder:

```bash
cd /path/to/DecoupliWo
tar -cf transitions.tar -C data transitions
```

Then upload `transitions.tar` to Google Drive at:

```
MyDrive/DecoupliWo/transitions.tar
```

(Uploading a single tar is ~10–100× faster than uploading millions of shard files individually.)

The notebook will extract it to the Colab VM's local disk on first run (training reads from local disk, **not** Drive, to avoid I/O stalls).

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Paths (edit here if yours differ)

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/wessimpson/DecoupliWo.git'
REPO_BRANCH = 'gameNGen-world_model-v2'
REPO_DIR = Path('/content/DecoupliWo')

DRIVE_ROOT = Path('/content/drive/MyDrive/DecoupliWo')
DRIVE_TAR = DRIVE_ROOT / 'transitions.tar'
DRIVE_CKPT_DIR = DRIVE_ROOT / 'checkpoints' / 'ascii_vae'
DRIVE_RUNS_DIR = DRIVE_ROOT / 'runs' / 'ascii_vae'

LOCAL_DATA_DIR = REPO_DIR / 'data' / 'transitions'

DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)

assert DRIVE_TAR.is_file(), (
    f'Expected {DRIVE_TAR} on Drive. Tar your local data/transitions/ '
    f'folder and upload it to MyDrive/DecoupliWo/transitions.tar first.'
)
print('OK', DRIVE_TAR, f'({DRIVE_TAR.stat().st_size / 1e9:.2f} GB)')

## 4. Clone the repo

In [ ]:
import subprocess

if REPO_DIR.is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(
        ['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
print('cwd =', os.getcwd())
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)

## 5. Install minimal dependencies

The full `requirements.txt` drags in Atari / OCAtari / stable-baselines3 / diffusers / etc. None of those are needed to train `ASCIIVAE`. We only need `numpy`, `tensorboard`, `tqdm`, and PyTorch (pre-installed on Colab GPU runtimes).

In [ ]:
!pip install --quiet 'numpy<2' tensorboard tqdm

## 6. Extract transitions tar onto the Colab VM's local disk

Reading from `/content/drive/...` during training is extremely slow (network round-trips per mmap page). Extracting to the VM's SSD once and training from there is dramatically faster.

In [ ]:
import shutil
import tarfile

LOCAL_DATA_DIR.parent.mkdir(parents=True, exist_ok=True)

if LOCAL_DATA_DIR.exists() and any(LOCAL_DATA_DIR.iterdir()):
    print(f'already extracted -> {LOCAL_DATA_DIR}')
else:
    if LOCAL_DATA_DIR.exists():
        shutil.rmtree(LOCAL_DATA_DIR)
    with tarfile.open(DRIVE_TAR, 'r') as tf:
        tf.extractall(LOCAL_DATA_DIR.parent)
    print(f'extracted -> {LOCAL_DATA_DIR}')

print('train shards:', len(list((LOCAL_DATA_DIR / 'train').rglob('shard_*'))))
print('test  shards:', len(list((LOCAL_DATA_DIR / 'test').rglob('shard_*'))))

## 7. Sanity check: one forward pass

In [ ]:
import sys
import torch

sys.path.insert(0, str(REPO_DIR))

from world_model.ascii.constants import CANVAS_H, CANVAS_W, VOCAB_SIZE
from world_model.model.net.ascii_vae import ASCIIVAE
from world_model.train_ascii_vae import AllAsciiFramesDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ds = AllAsciiFramesDataset(LOCAL_DATA_DIR / 'train', CANVAS_H, CANVAS_W)
print(f'train frames = {len(ds):,}   canvas = {CANVAS_H}x{CANVAS_W}   vocab = {VOCAB_SIZE}')

vae = ASCIIVAE().to(device)
ids = torch.stack([ds[i] for i in range(4)]).to(device)
logits, mu, logvar = vae(ids)
print('ids    ', tuple(ids.shape), ids.dtype)
print('logits ', tuple(logits.shape))
print('mu     ', tuple(mu.shape))
print('params ', sum(p.numel() for p in vae.parameters()))

## 8. Launch TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{DRIVE_RUNS_DIR}"

## 9. Train

Notes on the flags:

- `--output_dir` and `--log_dir` point at Drive so checkpoints and TensorBoard logs survive VM termination.
- `--batch_size 256` + `--num_workers 4` keeps the small H100/A100 fed without overwhelming the tiny model.
- `--save_every 5000` writes `vae.pt` to Drive every 5k steps as a crash-safety net (trainer also saves at the end and sets `scaling_factor` there).

In [ ]:
!python -m world_model.train_ascii_vae \
  --train_dir "{LOCAL_DATA_DIR}/train" \
  --val_dir   "{LOCAL_DATA_DIR}/test"  \
  --output_dir "{DRIVE_CKPT_DIR}" \
  --log_dir    "{DRIVE_RUNS_DIR}" \
  --batch_size 256 \
  --num_workers 4 \
  --epochs 4 \
  --max_train_steps 50000 \
  --save_every 5000 \
  --validation_every 1000 \
  --log_every 50

## 10. Verify saved checkpoint

In [ ]:
from world_model.model.net.ascii_vae import load_ascii_vae

ckpt = DRIVE_CKPT_DIR / 'vae.pt'
assert ckpt.is_file(), f'no checkpoint at {ckpt}'

vae = load_ascii_vae(ckpt).to(device)
vae.train(False)
ids = torch.stack([ds[i] for i in range(4)]).to(device)
logits, _, _ = vae(ids)
pred = ASCIIVAE.logits_to_ids(logits)
acc = (pred == ids).float().mean().item()
print(f'checkpoint    : {ckpt}  ({ckpt.stat().st_size / 1e6:.2f} MB)')
print(f'scaling_factor: {vae.scaling_factor:.4f}')
print(f'per-cell acc  : {acc:.4f} on 4 sample frames')